# Notebook 11 - Primary ROI Inference Ablation

Runs adapter inference only on the router primary SAM3-derived bbox ROI. Missing bbox rows are reported as `roi_missing` without full-image fallback.

In [ ]:
from pathlib import Path
import sys

def _ensure_aads_repo_on_path():
    candidates = [Path.cwd(), Path('/content/AADS-v6'), ]
    for candidate in candidates:
        if (candidate / 'src').is_dir() and (candidate / 'scripts').is_dir():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError('AADS repo root not found. Clone or mount the repo before running this notebook.')

ROOT = _ensure_aads_repo_on_path()
print(f'[BOOT] ROOT={ROOT}')


In [ ]:
from pathlib import Path
from scripts.colab_roi_ablation import ABLATION_CONFIGS, commit_and_push_ablation_results, run_ablation_folder

ABLATION_NAME = 'primary_roi_inference'
IMAGE_DIR = ROOT / 'data' / 'prepared_runtime_datasets' / 'tomato__fruit' / 'test'
ADAPTER_ROOT = ROOT / 'runs' / 'tomato' / 'fruit' / 'tomato_fruit_2026-05-26_10-34-27' / 'outputs' / 'colab_notebook_training'
CONFIG_ENV = 'colab'
DEVICE = 'cuda'
LABEL_FROM_PARENT = True
RETURN_OOD = True
OUTPUT_DIR = ROOT / ABLATION_CONFIGS[ABLATION_NAME]['output_dir']

print(f'[CONFIG] ablation={ABLATION_NAME} output={OUTPUT_DIR}')


In [ ]:
if not Path(IMAGE_DIR).is_dir():
    raise FileNotFoundError(f'IMAGE_DIR not found: {IMAGE_DIR}')
if not Path(ADAPTER_ROOT).is_dir():
    raise FileNotFoundError(f'ADAPTER_ROOT not found: {ADAPTER_ROOT}')

report = run_ablation_folder(
    IMAGE_DIR,
    ablation_name=ABLATION_NAME,
    output_dir=OUTPUT_DIR,
    label_from_parent=LABEL_FROM_PARENT,
    config_env=CONFIG_ENV,
    device=DEVICE,
    adapter_root=ADAPTER_ROOT,
    return_ood=RETURN_OOD,
)
git_result = commit_and_push_ablation_results(
    OUTPUT_DIR,
    repo_root=ROOT,
    message='Add primary ROI ablation results',
)
{'summary': report['summary'], 'git': git_result}
